# E3: LViT-T With text_lvit_prompt on Preprocessed BTXRD 224x224

This notebook trains the text-conditioned LViT baseline. E3 uses the same LViT Double-U image backbone as E2, plus HuggingFace BERT embeddings shaped `[B, 10, 768]` for `text_lvit_prompt`.

## 1. Paths And Run Settings

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/lehngoc/BTXRD-LViT.git"
BRANCH = "model/e2e3-lvit-224"
REPO_ROOT = Path("/kaggle/working/BTXRD-LViT")
CONFIG = REPO_ROOT / "configs/train_lvit_t_preprocessed_224.yaml"

DATA_ROOT = Path("/kaggle/input/datasets/lehngoc/btxrd-preprocessed-dataset/btxrd-preprocessed")
OUTPUT_DIR = Path("/kaggle/working/experiments/E3_lvit_t_preprocessed_224_tumor_only_h4")
SMOKE_OUTPUT_DIR = Path("/kaggle/working/experiments/_smoke_E3_lvit_t")

## 2. Clone The LViT Branch

In [ ]:
!rm -rf {REPO_ROOT}
!git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}
%cd {REPO_ROOT}
!git branch --show-current
!git rev-parse --short HEAD
!ls configs

if not CONFIG.exists():
    raise FileNotFoundError(f"Missing config after clone: {CONFIG}. Check that BRANCH={BRANCH!r} cloned successfully.")

## 3. Install Text Encoder Dependency

In [ ]:
# Kaggle Internet must be enabled. bert-base-uncased is downloaded on first model load.
!pip install -q transformers

## 4. Build Runtime Config

In [ ]:
import copy
import yaml

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}. Attach the BTXRD preprocessed Kaggle dataset or edit DATA_ROOT.")

cfg = yaml.safe_load(CONFIG.read_text())
cfg["data"]["root_dir"] = str(DATA_ROOT)
cfg["training"]["device"] = "cuda"
cfg["training"]["num_workers"] = 2
cfg["training"]["output_dir"] = str(OUTPUT_DIR)

runtime_config = Path("/kaggle/working/e3_lvit_t_runtime.yaml")
runtime_config.write_text(yaml.safe_dump(cfg, sort_keys=False))

smoke_cfg = copy.deepcopy(cfg)
smoke_cfg["training"]["output_dir"] = str(SMOKE_OUTPUT_DIR)
smoke_config = Path("/kaggle/working/e3_lvit_t_smoke.yaml")
smoke_config.write_text(yaml.safe_dump(smoke_cfg, sort_keys=False))

runtime_config, smoke_config

## 5. Smoke Test

In [ ]:
!rm -rf {SMOKE_OUTPUT_DIR}
!python -m src.training.train_lvit_t --config {smoke_config} --epochs 1 --max-train-samples 4 --max-val-samples 4 --max-test-samples 4 --device cuda

## 6. Full E3 Training

In [ ]:
!rm -rf {OUTPUT_DIR}
!python -m src.training.train_lvit_t --config {runtime_config} --device cuda

## 7. Export Metrics

In [ ]:
zip_path = Path("/kaggle/working/E3_lvit_t_preprocessed_224_h4_metrics_only.zip")
!cd {OUTPUT_DIR} && zip -r {zip_path} history.csv best_summary.json val_metrics.json test_metrics.json test_threshold_sweep_metrics.json test_metrics_thr*.json config.json
zip_path